# DBMF sizing

Research notebook for sizing a managed-futures sleeve against a baseline portfolio of `80% equity + 100% 10Y bond proxy + 10% commodity proxy`.

The key question is not standalone DBMF Sharpe. It is whether a DBMF / managed-futures proxy improves marginal risk contribution, monthly CVaR, max drawdown, and equity-bond selloff behavior.

Data notes:
- DBMF live history starts in 2019 and is pulled from Yahoo Finance.
- SG Trend Index / SG CTA Index are better long-history managed-futures proxies if licensed/downloaded locally. Societe Generale's current public page states full daily data access requires SG Markets / Capital Consulting; this notebook probes known public paths first, then looks for local files under `data/interim/` or `data/private/`.
- Any 9% volatility-normalized SG proxy is a research diagnostic, not a tradable backtest. The scaling uses full-sample proxy volatility to map the long SG history onto DBMF's stated 8-10% target-vol range.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from alpha_lab.analytics.returns import cumulative_returns, drawdown
from alpha_lab.analytics.risk import cov_matrix, cvar, risk_contributions
from alpha_lab.backtest.metrics import summary
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly
from alpha_lab.utils.paths import DATA_DIR

In [ ]:
START = "2000-01-01"
END = None

# ETF proxies keep the research reproducible with public Yahoo data. Replace IEF/DBC with
# local futures total-return series if available.
EQUITY_TICKER = "SPY"
BOND_TICKER = "IEF"
COMMODITY_TICKER = "DBC"
DBMF_TICKER = "DBMF"

CORE_WEIGHTS = {
    EQUITY_TICKER: 0.80,
    BOND_TICKER: 1.00,
    COMMODITY_TICKER: 0.10,
}

DBMF_WEIGHTS = [0.00, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20]

# Funding one dollar of DBMF by reducing 50c equity, 25c commodity, 25c duration.
# This matches the policy question better than adding DBMF as pure extra gross exposure.
FUNDING_WEIGHTS = {
    EQUITY_TICKER: 0.50,
    BOND_TICKER: 0.25,
    COMMODITY_TICKER: 0.25,
}

MF_TARGET_VOL = 0.09
MONTHLY_CVAR_Q = 0.05
ROLLING_WINDOW_MONTHS = 36
MIN_STRESS_MONTHS = 6

## SG Trend / CTA data attempt

Societe Generale's public index page currently describes SG Trend and SG CTA but does not expose a full historical daily file in the live HTML. The notebook still checks plausible public file paths and local files. Put a licensed/exported file in one of these locations to activate the long-history analysis:

- `data/interim/sg_trend_index.csv`
- `data/interim/sg_trend_index.xlsx`
- `data/private/sg_trend_index.csv`
- `data/private/sg_trend_index.xlsx`

Expected columns are flexible: one date column plus either a return column or a level/index column.

In [ ]:
SG_SOURCE_PAGE = "https://wholesale.banking.societegenerale.com/en/prime-services-indices/summary-page/168827-4/"
SG_CANDIDATE_URLS = [
    "https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/SG_Trend_Index_Historical.xls",
    "https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/SG_Trend_Historical.xls",
    "https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/Trend_Index_Historical.xls",
    "https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/SG_CTA_Index_Historical.xls",
]

SG_LOCAL_FILES = [
    DATA_DIR / "interim" / "sg_managed_futures_indices.csv",
    DATA_DIR / "interim" / "sg_trend_index.csv",
    DATA_DIR / "interim" / "sg_trend_index.xlsx",
    DATA_DIR / "private" / "sg_trend_index.csv",
    DATA_DIR / "private" / "sg_trend_index.xlsx",
]

In [ ]:
def _parse_sg_history(raw: pd.DataFrame) -> pd.Series:
    """Parse a loose SG-style history file into daily simple returns."""
    df = raw.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(how="all")

    date_col = next((c for c in df.columns if "date" in c.lower()), df.columns[0])
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()

    lower_cols = {c.lower(): c for c in df.columns}
    return_candidates = [
        c for key, c in lower_cols.items() if "return" in key or key in {"ror", "daily", "daily return"}
    ]
    level_candidates = [
        c for key, c in lower_cols.items() if "level" in key or "index" in key or "value" in key
    ]

    if return_candidates:
        series = pd.to_numeric(df[return_candidates[0]], errors="coerce")
        # Vendor files commonly store returns as percent units.
        if series.abs().quantile(0.95) > 0.5:
            series = series / 100.0
        return series.dropna().rename("SG Trend")

    if level_candidates:
        levels = pd.to_numeric(df[level_candidates[0]], errors="coerce")
    else:
        numeric = df.apply(pd.to_numeric, errors="coerce")
        levels = numeric.dropna(axis=1, how="all").iloc[:, 0]

    return levels.pct_change().dropna().rename("SG Trend")


def _read_history(path_or_url: str | Path) -> pd.Series:
    path_or_url = str(path_or_url)
    if path_or_url.lower().endswith((".xls", ".xlsx")):
        raw = pd.read_excel(path_or_url)
    else:
        raw = pd.read_csv(path_or_url)
    return _parse_sg_history(raw)


def load_sg_trend_returns() -> tuple[pd.Series | None, pd.DataFrame]:
    rows = []

    for url in SG_CANDIDATE_URLS:
        try:
            series = _read_history(url)
            if not series.empty:
                rows.append({"source": url, "status": "ok", "n": len(series)})
                return series, pd.DataFrame(rows)
        except Exception as exc:
            rows.append({"source": url, "status": type(exc).__name__, "n": 0})

    for path in SG_LOCAL_FILES:
        if not path.exists():
            rows.append({"source": str(path), "status": "missing", "n": 0})
            continue
        try:
            series = _read_history(path)
            if not series.empty:
                rows.append({"source": str(path), "status": "ok", "n": len(series)})
                return series, pd.DataFrame(rows)
        except Exception as exc:
            rows.append({"source": str(path), "status": type(exc).__name__, "n": 0})

    return None, pd.DataFrame(rows)

In [ ]:
sg_returns, sg_source_log = load_sg_trend_returns()
sg_source_log

In [ ]:
tickers = [EQUITY_TICKER, BOND_TICKER, COMMODITY_TICKER, DBMF_TICKER]
prices = load_prices(tickers, start=START, end=END).dropna(how="all")
daily_returns = prices.pct_change().dropna(how="all")
prices.tail()

In [ ]:
dbmf_returns = daily_returns[DBMF_TICKER].dropna().rename("DBMF")

if sg_returns is not None:
    raw_mf_returns = sg_returns.rename("SG Trend raw")
    source_label = "SG Trend"
else:
    raw_mf_returns = dbmf_returns.rename("DBMF live")
    source_label = "DBMF live only"

realized_mf_vol = raw_mf_returns.std() * np.sqrt(252)
mf_returns = (raw_mf_returns * (MF_TARGET_VOL / realized_mf_vol)).rename("Managed futures 9 vol")

pd.Series(
    {
        "source": source_label,
        "start": raw_mf_returns.index.min(),
        "end": raw_mf_returns.index.max(),
        "n_daily_returns": len(raw_mf_returns),
        "raw_ann_vol": realized_mf_vol,
        "normalized_ann_vol": mf_returns.std() * np.sqrt(252),
    }
)

## Portfolio construction

Default sizing is funded rather than pure overlay. A 10% DBMF sleeve is funded by reducing equity 5%, commodities 2.5%, and the 10Y bond proxy 2.5%. Change `FUNDING_WEIGHTS` above to test other funding policies.

In [ ]:
def funded_weights(mf_weight: float) -> pd.Series:
    weights = pd.Series(CORE_WEIGHTS, dtype=float)
    funding = pd.Series(FUNDING_WEIGHTS, dtype=float)
    weights = weights.sub(funding * mf_weight, fill_value=0.0)
    weights["MF"] = mf_weight
    return weights


asset_returns = daily_returns[[EQUITY_TICKER, BOND_TICKER, COMMODITY_TICKER]].join(mf_returns.rename("MF"), how="inner")
portfolio_returns = {}
candidate_weights = {}

for mf_weight in DBMF_WEIGHTS:
    weights = funded_weights(mf_weight)
    name = f"{mf_weight:.1%} MF"
    candidate_weights[name] = weights
    portfolio_returns[name] = fixed_weight_returns(asset_returns, weights)

portfolio_returns = pd.DataFrame(portfolio_returns).dropna(how="any")
candidate_weights = pd.DataFrame(candidate_weights).T
candidate_weights

In [ ]:
portfolio_returns.tail()

## Decision table

Pick the highest managed-futures allocation where monthly CVaR and max drawdown improve, while MF risk contribution remains below the policy cap. The stress columns use monthly returns, because daily noise can hide the portfolio-level role.

In [ ]:
def monthly_returns(returns: pd.Series | pd.DataFrame) -> pd.Series | pd.DataFrame:
    return (1 + returns).resample("ME").prod() - 1


monthly_assets = monthly_returns(asset_returns)
monthly_portfolios = monthly_returns(portfolio_returns)

equity_crash = monthly_assets[EQUITY_TICKER] < -0.05
bond_shock = monthly_assets[BOND_TICKER] < -0.03
equity_bond_down = (monthly_assets[EQUITY_TICKER] < 0) & (monthly_assets[BOND_TICKER] < 0)
combined_core_shock = (monthly_assets[EQUITY_TICKER] + monthly_assets[BOND_TICKER]).rank(pct=True) <= 0.10
equity_bond_selloff = equity_bond_down & combined_core_shock

stress_counts = pd.Series(
    {
        "equity_crash_months": int(equity_crash.sum()),
        "bond_shock_months": int(bond_shock.sum()),
        "equity_bond_down_months": int(equity_bond_down.sum()),
        "equity_bond_selloff_months": int(equity_bond_selloff.sum()),
    }
)
stress_counts

In [ ]:
def stress_mean(returns: pd.DataFrame, mask: pd.Series) -> pd.Series:
    mask = mask.reindex(returns.index).fillna(False)
    if int(mask.sum()) < MIN_STRESS_MONTHS:
        return pd.Series(np.nan, index=returns.columns)
    return returns.loc[mask].mean()


def mf_risk_contribution_pct(weights: pd.Series, returns: pd.DataFrame) -> float:
    cov = cov_matrix(returns.reindex(columns=weights.index).dropna(), periods=252)
    rc = risk_contributions(weights, cov)
    total = rc.sum()
    return float(rc.get("MF", 0.0) / total) if total != 0 else np.nan


stats = pd.DataFrame({name: pd.Series(summary(series)) for name, series in portfolio_returns.items()}).T
stats["MonthlyCVaR5"] = monthly_portfolios.apply(cvar, q=MONTHLY_CVAR_Q)
stats["Worst1M"] = monthly_portfolios.min()
stats["EquityCrashAvg"] = stress_mean(monthly_portfolios, equity_crash)
stats["BondShockAvg"] = stress_mean(monthly_portfolios, bond_shock)
stats["EquityBondDownAvg"] = stress_mean(monthly_portfolios, equity_bond_down)
stats["WorstEquityBondSelloffAvg"] = stress_mean(monthly_portfolios, equity_bond_selloff)
stats["CorrToEquity"] = portfolio_returns.corrwith(asset_returns[EQUITY_TICKER])
stats["CorrTo10YProxy"] = portfolio_returns.corrwith(asset_returns[BOND_TICKER])
stats["MFRiskContribution"] = [
    mf_risk_contribution_pct(candidate_weights.loc[name], asset_returns) for name in stats.index
]
stats["FinalWealth"] = cumulative_returns(portfolio_returns).iloc[-1]

decision_cols = [
    "CAGR",
    "AnnVol",
    "Sharpe",
    "MaxDD",
    "MonthlyCVaR5",
    "WorstEquityBondSelloffAvg",
    "CorrToEquity",
    "CorrTo10YProxy",
    "MFRiskContribution",
    "FinalWealth",
]
stats[decision_cols]

In [ ]:
base = stats.loc["0.0% MF", decision_cols]
delta_vs_base = stats[decision_cols].sub(base, axis=1)
delta_vs_base

In [ ]:
eligible = stats[
    (stats["MonthlyCVaR5"] > stats.loc["0.0% MF", "MonthlyCVaR5"])
    & (stats["MaxDD"] > stats.loc["0.0% MF", "MaxDD"])
    & (stats["MFRiskContribution"] <= 0.15)
]

pd.Series(
    {
        "rule": "highest weight where CVaR and MaxDD improve and MF risk contribution <= 15%",
        "candidate": eligible.index[-1] if not eligible.empty else "no allocation passes all gates",
        "source": source_label,
    }
)

## Conditional managed-futures behavior

These rows answer the bad-times-beta question directly. If the managed-futures proxy does not help during equity-bond down months, keep sizing closer to 5-10% even if full-sample correlation looks low.

In [ ]:
mf_monthly = monthly_assets["MF"]
conditional_mf = pd.Series(
    {
        "all_months": mf_monthly.mean(),
        "equity_crash": mf_monthly.loc[equity_crash].mean(),
        "bond_shock": mf_monthly.loc[bond_shock].mean(),
        "equity_bond_down": mf_monthly.loc[equity_bond_down].mean(),
        "worst_equity_bond_selloff": mf_monthly.loc[equity_bond_selloff].mean(),
    }
)
conditional_mf

In [ ]:
rolling_corr = monthly_assets["MF"].rolling(ROLLING_WINDOW_MONTHS).corr(monthly_assets[EQUITY_TICKER])
rolling_corr.to_frame("MF rolling corr to equity").join(
    monthly_assets["MF"].rolling(ROLLING_WINDOW_MONTHS).corr(monthly_assets[BOND_TICKER]).rename("MF rolling corr to 10Y proxy")
).plot(title=f"Rolling {ROLLING_WINDOW_MONTHS}M managed-futures correlations")

## Charts

In [ ]:
equity_curve(portfolio_returns)

In [ ]:
drawdown(portfolio_returns).plot(title="Drawdowns by managed-futures allocation")

In [ ]:
best_by_rule = eligible.index[-1] if not eligible.empty else stats["Sharpe"].idxmax()
drawdown_chart(portfolio_returns[best_by_rule])

In [ ]:
heatmap_monthly(portfolio_returns[best_by_rule])

## Policy interpretation

Use this notebook to maintain the policy range:

- Strategic target: 10% managed futures.
- Normal range: 5-15%.
- Move toward 15% only if the stress rows improve and MF risk contribution stays below 15%.
- Avoid 20%+ unless trend following is intentionally promoted to a core risk sleeve rather than a diversifier.

Before changing actual sizing, check DBMF's current holdings / inferred positions so the sleeve is not silently doubling existing duration, commodity, or equity exposure.